[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q2/04_categorization.ipynb)

# R2-Q2 Week 4 — Apply the taxonomy and report the failure-mode distribution

This notebook applies the four numeric predicates committed in Week 1 to the Grad-CAM heatmaps produced in Week 3, assigning each misclassification to one of five categories — symptom-attended-but-wrong-class, background-attended, lighting-attended, leaf-shape-attended, or other/ambiguous — following the first-match-wins scheme with multi-fires routing to "other." The output is the failure-mode distribution across the misclassification set, reported at the overall taxonomy level rather than per-class, and the methods-section text documenting how each predicate was operationalized.

By the end of this notebook you will have:

- A `categorizations.parquet` containing one row per misclassification with columns recording which predicates fired, the assigned category, and the derived quantities (m_out, m_perimeter, lighting_correlation, m_in, concentration) that drove the assignment — so a reader can audit any individual categorization.
- A `taxonomy_distribution.json` recording the count and fraction of misclassifications in each of the five categories, the conjunction-handling statistics (how many images fired multiple predicates and routed to "other"), and the headline interpretation sentence describing what the failure-mode distribution looks like.
- A `taxonomy_distribution.png` bar chart showing the five-category distribution with the "other" bucket visually distinguished — so a reader can see at a glance whether failures clustered into recognizable patterns or whether the taxonomy itself needs revision.

## Before you run anything: switch to a GPU runtime

This notebook uses a large vision model (the Segment Anything Model,
or SAM) to separate the leaf from the background in each of the 81
misclassified images. SAM runs much faster on a GPU than on a CPU,
and 81 images on CPU adds up to a long wait. Colab's default runtime
is CPU-only, so you'll need to switch.

To switch:

1. In Colab's menu bar: **Runtime → Change runtime type**.
2. Under "Hardware accelerator," choose **T4 GPU**.
3. Click **Save**.

Wait for the session to come up on the new runtime (a few seconds).
Then run the cells below in order.

**If Colab tells you "no GPUs available right now":** free-tier GPU
access is shared and occasionally unavailable. Wait a few minutes
and try again. If it keeps refusing, tell your mentor — the
upgraded Colab tiers have more reliable GPU access.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in the following Cell:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in the following Cell:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

### Confirm the GPU is in place

Before going further, confirm Colab attached a GPU. The cell below
prints `GPU detected:` followed by `True` or `False`. If it's `True`,
you're good. If it's `False`, go back to the top of the notebook and
re-do the runtime switch — the next cells will not work without a GPU.

The check is a separate cell from the main setup (below) because it's
fast and recoverable: if the GPU isn't there, you see the problem
right away with no Drive authorization prompt to dismiss first.

In [ ]:
import irilab2026 as iri

gpu_ok = iri.has_gpu()
print(f"GPU detected: {gpu_ok}")

assert gpu_ok, (
    "No GPU detected. Go to the top of the notebook and follow the "
    "runtime-switch instructions, then re-run from Setup Step 1."
)

In [ ]:
# Mount Drive and set up irilab2026 with a GPU.
import irilab2026 as iri
iri.setup(gpu_required=True)

OUTPUT_DIR = iri.output_dir("r2_q2")
print(f"Output directory: {OUTPUT_DIR}")

## 1) Load the inputs from the earlier notebooks

This notebook reads three files produced by the earlier notebooks in
this series:

- `precommit.json`, from `01_pre_commits.ipynb`
- `working_set.parquet`, from `02_load_and_filter.ipynb`
- `gradcam_metadata.parquet`, from `03_sanity_checks_and_gradcam.ipynb`

The three subsections below load each in turn, with the validation
needed to catch the most common ways they can be missing or wrong.
All three loads share the same posture: try to read the file, raise
a helpful error if it's missing, validate just enough to catch the
obvious failures, and print what got loaded so you can see the
structure without inspecting it manually.

Deeper validation of the precommit's internal structure happens in
Sections 3, 4, and 5 — at the point where each sub-field is actually
used. That keeps error messages close to the symptoms that produce
them.

### 1.1) Load and validate the pre-commit

The precommit is the contract from Week 1 — the document that captures
every decision about how this categorization will run. N04 reads from
it in every later section: the leaf-segmentation method (Section 3),
the five derived quantities (Section 4), the four predicates with
their thresholds and order (Section 5), and the aggregation level for
the final summary (Section 6).

The validation below is intentionally light. It confirms the four
top-level fields the precommit promises (`taxonomy`,
`categorization_procedure`, `aggregation_level`, `sanity_checks`) —
enough to say "this is a real precommit, not an empty file" — plus
one slightly deeper check that the taxonomy's category list is
populated, since every later section iterates over it. Validation
of specific sub-fields (the thresholds, the predicate order, the
segmentation parameters) happens in the section that reads each one.

In [ ]:
### 1.1) Load and validate the pre-commit

import json

precommit_path = OUTPUT_DIR / "precommit.json"

try:
    with open(precommit_path) as f:
        precommit = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {precommit_path}. "
        "This file is produced by 01_pre_commits.ipynb — "
        "run that notebook to completion before running this one."
    ) from None

# Light validation: the four locked top-level keys must all be present.
# Each later section validates the specific sub-fields it uses.
expected_keys = {
    "taxonomy",
    "categorization_procedure",
    "sanity_checks",
    "aggregation_level",
}
missing = expected_keys - set(precommit.keys())
if missing:
    raise KeyError(
        f"precommit.json is missing top-level keys: {sorted(missing)}. "
        "This usually means N01 didn't complete — "
        "re-run 01_pre_commits.ipynb to regenerate the file."
    )

# Slightly deeper: the taxonomy categories list is what every later
# section iterates over. Check here that it's present and non-empty
# so the print below doesn't crash with a noisy KeyError.
if not precommit["taxonomy"].get("categories"):
    raise KeyError(
        "precommit['taxonomy']['categories'] is missing or empty. "
        "N01 Section 4 didn't complete — re-run 01_pre_commits.ipynb."
    )

print(f"Loaded: {precommit_path}")
print(f"  top-level keys: {sorted(precommit.keys())}")
print(f"  taxonomy ({len(precommit['taxonomy']['categories'])} categories):")
for cat in precommit["taxonomy"]["categories"]:
    print(f"    - {cat}")

### 1.2) Load the working set

N02 wrote `working_set.parquet` to this question's output directory:
the rows of the PlantDoc prediction table where the model
misclassified at the category level. N03 read this same file to
generate the Grad-CAM heatmaps; N04 reads it again to pair each
misclassification with its heatmap and its source image.

This notebook uses two columns from the working set: `filename` (the
per-image identifier that joins each row to its heatmap in the next
subsection) and `image_path` (the disk location of the source image,
needed for SAM segmentation in Section 3 and brightness analysis in
Section 4). Other columns pass through to the per-image output table
N04 produces in Section 6, so a reader can audit any individual
categorization against the original prediction data.

In [ ]:
### 1.2) Load the working set

import pandas as pd

working_set_path = OUTPUT_DIR / "working_set.parquet"

try:
    working_set = pd.read_parquet(working_set_path)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {working_set_path}. "
        "This file is produced by 02_load_and_filter.ipynb — "
        "run that notebook to completion before running this one."
    ) from None

# Light schema validation: the two columns N04 strictly depends on.
required_columns = {"filename", "image_path"}
missing = required_columns - set(working_set.columns)
if missing:
    raise KeyError(
        f"working_set.parquet is missing required columns: {sorted(missing)}. "
        "This usually means N02 wrote an older or partial schema — "
        "re-run 02_load_and_filter.ipynb to regenerate the file."
    )

# An empty working set means there are no misclassifications to
# categorize — methodologically interesting but operationally a stop
# condition for N04.
if len(working_set) == 0:
    raise ValueError(
        "working_set.parquet has zero rows. "
        "There are no misclassifications to categorize — "
        "if this is unexpected, check N02's filter logic."
    )

print(f"Loaded: {working_set_path}")
print(f"  rows: {len(working_set)}")
print(f"  columns: {sorted(working_set.columns)}")

### 1.3) Load the Grad-CAM metadata

N03 wrote one `.npy` file per misclassification — 81 heatmaps at 7×7
resolution, stored under `OUTPUT_DIR / "heatmaps"`. Alongside them
N03 wrote `gradcam_metadata.parquet`, a table that joins each heatmap
file back to its working-set row. Section 4 will load each heatmap
as it processes the corresponding image; this section confirms the
metadata is well-formed and the files are where the metadata says
they are.

The validation here is more aggressive than the previous two loads
because the file-existence check is genuinely cheap — 81 stat calls
take well under a second — and catching a missing heatmap here
saves the better part of a minute of SAM segmentation in Section 3
before the error would otherwise surface in Section 4.

Three checks, in order:

1. Every required column is present.
2. Every working-set row has a matching entry in the metadata.
3. Every referenced `.npy` file actually exists on disk.

In [ ]:
### 1.3) Load the Grad-CAM metadata

from pathlib import Path

gradcam_metadata_path = OUTPUT_DIR / "gradcam_metadata.parquet"

try:
    gradcam_metadata = pd.read_parquet(gradcam_metadata_path)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {gradcam_metadata_path}. "
        "This file is produced by 03_sanity_checks_and_gradcam.ipynb — "
        "run that notebook to completion before running this one."
    ) from None

if len(gradcam_metadata) == 0:
    raise ValueError(
        f"{gradcam_metadata_path} has zero rows. "
        "Re-run 03_sanity_checks_and_gradcam.ipynb to regenerate the file."
    )

# Check 1: required columns.
required_columns = {"filename", "heatmap_path"}
missing = required_columns - set(gradcam_metadata.columns)
if missing:
    raise KeyError(
        f"gradcam_metadata.parquet is missing required columns: "
        f"{sorted(missing)}. This usually means N03 wrote an older or "
        "partial schema — re-run 03_sanity_checks_and_gradcam.ipynb "
        "to regenerate the file."
    )

# Check 2: every working-set row has a matching heatmap entry.
working_filenames = set(working_set["filename"])
heatmap_filenames = set(gradcam_metadata["filename"])
missing_entries = working_filenames - heatmap_filenames
if missing_entries:
    raise KeyError(
        f"{len(missing_entries)} working-set row(s) have no entry in "
        f"gradcam_metadata. First few: {sorted(missing_entries)[:3]}. "
        "Re-run 03_sanity_checks_and_gradcam.ipynb to regenerate the "
        "metadata."
    )

# Check 3: every referenced .npy file exists on disk. Paths may be
# stored as absolute or relative; resolve relative paths against
# OUTPUT_DIR before checking.
missing_files = []
for p in gradcam_metadata["heatmap_path"]:
    resolved = Path(p)
    if not resolved.is_absolute():
        resolved = OUTPUT_DIR / resolved
    if not resolved.exists():
        missing_files.append(str(resolved))

if missing_files:
    raise FileNotFoundError(
        f"{len(missing_files)} heatmap file(s) referenced in "
        f"gradcam_metadata are missing from disk. "
        f"First few: {missing_files[:3]}. "
        "Re-run 03_sanity_checks_and_gradcam.ipynb to regenerate the "
        "heatmaps."
    )

print(f"Loaded: {gradcam_metadata_path}")
print(f"  rows: {len(gradcam_metadata)}")
print(f"  all {len(working_set)} working-set rows have matching heatmaps — OK")
print(f"  all {len(gradcam_metadata)} heatmap files present on disk — OK")